# Clase 4 — Chain-of-thought y prompts complejos

Hay tareas que requieren razonamiento en varios pasos: resolver un problema, tomar una decisión con múltiples criterios o analizar una situación con información incompleta. En esos casos, pedirle al modelo que piense en voz alta — **chain-of-thought** — mejora significativamente la precisión.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Qué es chain-of-thought y por qué funciona |
| 3 | CoT implícito vs. CoT explícito |
| 4 | CoT estructurado con pasos definidos |
| 5 | Restricciones y rúbricas dentro del prompt |
| 6 | Actividad: optimizar un prompt complejo |

---
## 1. Configuración del entorno

**Si es tu primera vez en este curso:**
1. Obtené tu API key en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. Guardala en `.env`:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. Si no querés crear el archivo, la celda te la pide de forma interactiva.

In [2]:
import os
import getpass

BACKEND = "ollama"         # "gemini", "ollama", "local"
GEMINI_MODEL = "gemini-2.5-flash-lite"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

Backend: ollama


In [3]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

elif BACKEND == "ollama":
    import ollama
    OLLAMA_MODEL = "gemma2:9b"  # Modelo que tenés descargado
    print("🚀 Conectando a Ollama...")
    try:
        ollama.list()
        print(f"✅ Ollama disponible. Usando modelo: {OLLAMA_MODEL}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Solución: Abre otra terminal y ejecuta: ollama serve")
        raise

elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        filename="qwen2.5-0.5b-instruct-q4_k_m.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=300):
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    elif BACKEND == "ollama":
        r = ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            system=system_prompt,
            stream=False,
            options={
                "temperature": temperature,
                "num_predict": max_tokens,
            }
        )
        return r['response'].strip()
    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

🚀 Conectando a Ollama...
✅ Ollama disponible. Usando modelo: gemma2:9b
Entorno listo.


---
## 2. Qué es chain-of-thought y por qué funciona

Cuando un modelo responde directamente una pregunta compleja, puede saltarse pasos lógicos y llegar a una conclusión incorrecta. **Chain-of-thought (CoT)** es la técnica de pedirle que muestre su razonamiento paso a paso antes de dar la respuesta final.

Funciona porque:
- El modelo genera cada token en función de los anteriores. Si los pasos intermedios están escritos, el siguiente token es más probable que sea correcto.
- El razonamiento visible también te permite detectar dónde se equivocó.

| Tipo de tarea | ¿Vale la pena usar CoT? |
|---|---|
| Saludo, formato simple, resumen breve | No — el overhead no aporta |
| Razonamiento lógico o matemático | Sí — mejora mucho la precisión |
| Decisiones con múltiples criterios | Sí — fuerza a ponderar antes de concluir |
| Análisis de situaciones ambiguas | Sí — reduce conclusiones apresuradas |

---
## 3. CoT implícito vs. CoT explícito

Hay dos formas de activar el razonamiento paso a paso:

In [8]:
# ─── Tarea de referencia ─────────────────────────────────────────────────────
# Una empresa tiene tres candidatos para un puesto. Necesita elegir uno
# considerando: experiencia, disponibilidad y presupuesto.

caso = """Una empresa necesita contratar un analista de datos.
Tiene tres candidatos:
- Ana: 5 años de experiencia, disponible en 2 semanas, pide $2.500/mes.
- Bruno: 2 años de experiencia, disponible de inmediato, pide $1.800/mes.
- Carmen: 8 años de experiencia, disponible en 2 meses, pide $3.200/mes.
El proyecto empieza en 3 semanas y el presupuesto máximo es $2.800/mes."""

pregunta = "¿A quién deberían contratar y por qué?"

# ─── Sin CoT: respuesta directa ──────────────────────────────────────────────
print("=== SIN CoT ===")
print(llamar_llm(f"{caso}\n\n{pregunta}", max_tokens=1000))
print()

=== SIN CoT ===
Deben contratar a **Ana**.  

Aquí está el razonamiento:

* **Experiencia:** Ana tiene 5 años de experiencia, lo cual es suficiente para el puesto.
* **Disponibilidad:**  Puede empezar en 2 semanas, coincidiendo con la fecha de inicio del proyecto.
* **Presupuesto:** Su salario de $2.500/mes está dentro del presupuesto máximo.


Bruno tiene menos experiencia y Carmen, aunque experimentada, no estaría disponible a tiempo.



In [5]:
# ─── CoT implícito: la frase mágica ──────────────────────────────────────────
# Solo agregar "Pensá paso a paso" activa el razonamiento en muchos casos.

print("=== CoT IMPLÍCITO ===")
prompt_cot_implicito = f"{caso}\n\n{pregunta}\n\nPensá paso a paso antes de responder."
print(llamar_llm(prompt_cot_implicito, max_tokens=1000))
print()

=== CoT IMPLÍCITO ===
Aquí está el análisis paso a paso para determinar la mejor opción:

1. **Prioridades:** La empresa necesita un analista de datos disponible en 3 semanas con un presupuesto máximo de $2.800/mes. 

2. **Evaluación de candidatos:**
    * **Ana:** Disponible en 2 semanas, experiencia (5 años) y salario ($2.500/mes) encajan dentro del presupuesto y plazo. Es la opción ideal si no hay problemas con su nivel de experiencia.
    * **Bruno:** Disponible inmediatamente, pero solo tiene 2 años de experiencia. Su salario ($1.800/mes) es atractivo, pero su falta de experiencia podría ser un riesgo.
    * **Carmen:** Tiene mucha experiencia (8 años), pero no está disponible por 2 meses.  Su salario ($3.200/mes) supera el presupuesto.

3. **Conclusión:**

La mejor opción es **Ana**. Cumple con los requisitos de disponibilidad y presupuesto, además de tener una cantidad significativa de experiencia.



In [12]:
# ─── CoT explícito: pasos definidos por nosotros ─────────────────────────────
# Le indicamos exactamente qué analizar en cada paso.

print("=== CoT EXPLÍCITO ===")
prompt_cot_explicito = f"""{caso}

Para elegir al candidato, seguí estos pasos en orden:
Paso 1: Elimina a los candidatos que piden mas que el presupuesto.
Paso 1.1: lista los que que piden menos que el presupuesto.
Paso 2: De la lista previamente filtrada, descartá a quienes no pueden empezar a tiempo.
Paso 2.1: lista los que pueden empezar a tiempo.
Paso 3: De los que quedan, elegí el de mayor experiencia.
Paso 4: Escribí la recomendación final que cumpla con los requerimientosen una oración."""


print(llamar_llm(prompt_cot_explicito, max_tokens=300))

=== CoT EXPLÍCITO ===
## Selección del Analista de Datos

**Paso 1:**  Los candidatos que piden menos de $2.800/mes son Bruno ($1.800) y Ana ($2.500).

**Paso 2:** De la lista, los que pueden empezar en 3 semanas son Bruno (disponible inmediatamente) y Ana (en 2 semanas).

**Paso 3:** El candidato con mayor experiencia es Carmen (8 años), pero su salario supera el presupuesto.  Por lo tanto, entre Bruno y Ana, Ana tiene más experiencia (5 años).


**Recomendación final:** Se recomienda contratar a Ana, ya que cumple con los requisitos de experiencia, disponibilidad y presupuesto.


> 💡 **Para discutir:** ¿Cuál de los tres enfoques fue más útil para este caso? ¿En qué tipo de decisiones de tu trabajo aplicarías CoT explícito?

---
## 4. CoT estructurado con pasos definidos

Cuando la tarea se repite muchas veces (por ejemplo, analizar contratos, evaluar propuestas, revisar reportes), conviene estandarizar los pasos del CoT en una plantilla reutilizable.

In [13]:
# ─── Plantilla CoT para análisis de riesgo ───────────────────────────────────
# Caso de uso: evaluar si un proveedor nuevo es confiable.

def analizar_proveedor_cot(descripcion_proveedor):
    prompt = f"""Sos un analista de compras evaluando proveedores nuevos.

Información del proveedor:
{descripcion_proveedor}

Analizá siguiendo estos pasos:
Paso 1 — Experiencia: ¿Cuántos años opera y en qué industrias?
Paso 2 — Capacidad: ¿Puede cumplir el volumen y los plazos requeridos?
Paso 3 — Riesgos: Identificá hasta 2 riesgos concretos.
Paso 4 — Recomendación: Aprobado / Aprobado con condiciones / Rechazado + una oración de justificación.

Mostrá cada paso claramente numerado."""

    return llamar_llm(prompt, max_tokens=350)


proveedor_ejemplo = """TechSupply S.A. lleva 3 años en el mercado de insumos electrónicos.
Tiene capacidad para entregar 500 unidades mensuales con un plazo de 15 días.
No tiene certificaciones de calidad pero sí referencias de dos clientes medianos.
Su precio es un 20% menor al mercado actual."""

print(analizar_proveedor_cot(proveedor_ejemplo))

## Análisis de Proveedor: TechSupply S.A.

**Paso 1 — Experiencia:** TechSupply S.A. opera en el mercado de insumos electrónicos desde hace **3 años**.  No se menciona su experiencia en otras industrias.

**Paso 2 — Capacidad:** Puede cumplir con la capacidad requerida (500 unidades mensuales) y el plazo de entrega (15 días).

**Paso 3 — Riesgos:**
* **Falta de certificaciones de calidad:** La ausencia de certificaciones puede indicar un menor control de calidad en sus procesos.
* **Referencias limitadas:**  Depender únicamente de referencias de dos clientes medianos podría no ser suficiente para evaluar su fiabilidad y capacidad de servicio a gran escala.

**Paso 4 — Recomendación:** **Aprobado con condiciones.** Si bien TechSupply S.A. cumple con los requisitos básicos, se recomienda solicitar pruebas de calidad y ampliar las referencias a clientes de mayor tamaño antes de formalizar una compra significativa.


---
## 5. Restricciones y rúbricas dentro del prompt

Para casos donde la salida necesita cumplir criterios específicos, podemos incluir una **rúbrica** dentro del prompt: una lista explícita de condiciones que la respuesta debe satisfacer. El modelo la usa como checklist interno.

In [14]:
# ─── Prompt con rúbrica explícita ─────────────────────────────────────────────
# Tarea: escribir un mensaje de feedback para un empleado.

situacion = """Lucas entregó el informe dos días tarde, pero la calidad del trabajo
fue muy buena. Es la primera vez que se retrasa en 18 meses de trabajo."""

prompt_con_rubrica = f"""Sos un líder de equipo. Escribí un mensaje de feedback para Lucas sobre esta situación:
{situacion}

El mensaje debe cumplir TODOS estos criterios:
✓ Reconocer primero lo positivo antes de mencionar el problema.
✓ Mencionar el retraso como un hecho, sin juicio sobre la persona.
✓ Proponer una acción concreta para evitar que se repita.
✓ Tono constructivo y directo, sin condescendencia.
✓ Máximo 5 oraciones en total.

Antes del mensaje, verificá en una línea que cumplís cada criterio."""

print(llamar_llm(prompt_con_rubrica, max_tokens=350))

✔ Reconocer lo positivo
✔ Mencionar el retraso como hecho
✔ Acción concreta
✔ Tono constructivo
✔ Máximo 5 oraciones


Lucas, el informe está excelente, realmente refleja un gran esfuerzo y la calidad es muy alta.  Se entregó dos días después del plazo acordado. Para evitar retrasos en el futuro, ¿te parece si nos avisas si hay algún imprevisto que pueda afectar la entrega a tiempo? Así podemos trabajar juntos para encontrar soluciones.


In [16]:
# ─── Comparación: sin rúbrica vs. con rúbrica ─────────────────────────────────

print("=== SIN RÚBRICA ===")
print(llamar_llm(f"Sos un líder de equipo. Escribí un mensaje de feedback para Lucas: {situacion}",
    max_tokens=200
))
print()
print("=== CON RÚBRICA ===")
print(llamar_llm(prompt_con_rubrica, max_tokens=350))

=== SIN RÚBRICA ===
Lucas,

El informe está excelente. El análisis y las conclusiones son precisas y bien presentadas. 

Si bien aprecio tu dedicación al proyecto, la entrega tuvo un retraso de dos días.  Es importante mantener el cronograma establecido para asegurar un flujo de trabajo eficiente en equipo. ¿Podrías compartir qué sucedió que provocó este retraso?

En general, estoy satisfecho con tu trabajo y confío en que esto será una excepción.

=== CON RÚBRICA ===
☐ Reconocer lo positivo primero
☐ Mencionar el retraso como un hecho
☐ Proponer acción concreta
☐ Tono constructivo y directo
☐ Máximo 5 oraciones


Lucas, me gustó mucho la calidad del informe, se nota el gran trabajo que realizaste. Sin embargo, llegó dos días después de la fecha acordada. Para evitar confusiones en el futuro, ¿te parece si nos avisas si vas a tener dificultades para cumplir con una entrega?


---
## 6. Actividad: optimizar un prompt complejo

Tomá una tarea de razonamiento o decisión de tu trabajo. Escribí primero un prompt simple y luego una versión con CoT explícito + rúbrica. Compará ambas respuestas.

In [17]:
# TODO: Describí una situación que requiera razonamiento o decisión
mi_situacion = """Necesito ascender a un vendedor a Jefe de Equipo de Ventas."""

# ─── Versión simple ───────────────────────────────────────────────────────────
# TODO: Escribí un prompt directo sin CoT
mi_prompt_simple = """Propon un candidato para ascender a Jefe de Equipo, teniendo en cuenta que los objetivos para el próximo semestre son:
 aumentar ventas un 30%, y capacitar a 3 vendedores nuevos.

Candidatos:

- **Martín**: 
  - Cumplimiento de cuota: 145% (promedio últimos 3 años)
  - Experiencia liderando: 0 años (siempre vendedor individual)
  - Relación con equipo: Excelente, comparte técnicas activamente
  - Velocidad de cierre: 45 días promedio
  - Años en la empresa: 6 años

- **Carolina**:
  - Cumplimiento de cuota: 120% (promedio últimos 3 años)
  - Experiencia liderando: 2 años liderando equipo en empresa anterior
  - Relación con equipo: Buena, a veces competitiva
  - Velocidad de cierre: 30 días promedio
  - Años en la empresa: 3 años

- **Roberto**:
  - Cumplimiento de cuota: 125% (promedio últimos 3 años)
  - Experiencia liderando: Mentor formal de 3 vendedores nuevos
  - Relación con equipo: Muy respetado, paciente
  - Velocidad de cierre: 60 días promedio
  - Años en la empresa: 8 años

- **Lucía**:
  - Cumplimiento de cuota: 135% (promedio últimos 3 años)
  - Experiencia liderando: Co-lideró proyecto interdepartamental 6 meses
  - Relación con equipo: Excelente, organiza capacitaciones voluntarias
  - Velocidad de cierre: 35 días promedio
  - Años en la empresa: 4 años

- **Diego**:
  - Cumplimiento de cuota: 110% (promedio últimos 3 años)
  - Experiencia liderando: 3 años como Team Lead en otra industria (retail)
  - Relación con equipo: Buena, muy estructurado y metódico
  - Velocidad de cierre: 50 días promedio
  - Años en la empresa: 2 años

- **Sofía**:
  - Cumplimiento de cuota: 130% (promedio últimos 3 años)
  - Experiencia liderando: 1 año liderando equipo de inside sales
  - Relación con equipo: Excelente, crea ambiente positivo
  - Velocidad de cierre: 40 días promedio
  - Años en la empresa: 5 años

"""

print("=== VERSIÓN SIMPLE ===")
print(llamar_llm(mi_prompt_simple, max_tokens=500))
print()

=== VERSIÓN SIMPLE ===
Basándome en los objetivos del próximo semestre (aumentar ventas un 30% y capacitar a 3 vendedores nuevos), el candidato más adecuado para ascender a Jefe de Equipo es **Roberto**. Aquí está mi razonamiento:

* **Experiencia en capacitación:** Roberto ya tiene experiencia formal como mentor de vendedores nuevos, lo cual es crucial para cumplir con el objetivo de capacitar a 3 vendedores.
* **Buen desempeño:** Con un promedio del 125% en cumplimiento de cuota, demuestra su capacidad para alcanzar objetivos de ventas.
* **Respeto y liderazgo:** Su actitud paciente y respetuosa le permitirá liderar un equipo efectivo y positivo.

Si bien otros candidatos como Martín o Lucía también tienen fortalezas, Roberto combina las habilidades necesarias para el rol: experiencia en entrenamiento, buen desempeño y liderazgo respetado.



In [5]:
# TODO: Reescribí el mismo prompt con pasos CoT explícitos y una rúbrica
mi_prompt_cot = """Necesito ascender a un vendedor a Jefe de Equipo de Ventas.

**Objetivos del próximo semestre:**
- Aumentar ventas un 30%
- Capacitar a 3 vendedores nuevos que ingresan al equipo

**Candidatos:**

- **Martín**: 
  - Cumplimiento de cuota: 145% (promedio últimos 3 años)
  - Experiencia liderando: 0 años (siempre vendedor individual)
  - Relación con equipo: Excelente, comparte técnicas activamente
  - Velocidad de cierre: 45 días promedio
  - Años en la empresa: 6 años

- **Carolina**:
  - Cumplimiento de cuota: 120% (promedio últimos 3 años)
  - Experiencia liderando: 2 años liderando equipo en empresa anterior
  - Relación con equipo: Buena, a veces competitiva
  - Velocidad de cierre: 30 días promedio
  - Años en la empresa: 3 años

- **Roberto**:
  - Cumplimiento de cuota: 125% (promedio últimos 3 años)
  - Experiencia liderando: Mentor formal de 3 vendedores nuevos
  - Relación con equipo: Muy respetado, paciente
  - Velocidad de cierre: 60 días promedio
  - Años en la empresa: 8 años

- **Lucía**:
  - Cumplimiento de cuota: 135% (promedio últimos 3 años)
  - Experiencia liderando: Co-lideró proyecto interdepartamental 6 meses
  - Relación con equipo: Excelente, organiza capacitaciones voluntarias
  - Velocidad de cierre: 35 días promedio
  - Años en la empresa: 4 años

- **Diego**:
  - Cumplimiento de cuota: 110% (promedio últimos 3 años)
  - Experiencia liderando: 3 años como Team Lead en otra industria (retail)
  - Relación con equipo: Buena, muy estructurado y metódico
  - Velocidad de cierre: 50 días promedio
  - Años en la empresa: 2 años

- **Sofía**:
  - Cumplimiento de cuota: 130% (promedio últimos 3 años)
  - Experiencia liderando: 1 año liderando equipo de inside sales
  - Relación con equipo: Excelente, crea ambiente positivo
  - Velocidad de cierre: 40 días promedio
  - Años en la empresa: 5 años

---

**Analizá siguiendo este proceso paso a paso:**

**PASO 1: Evaluar Performance en Ventas (0-10 puntos)**
Criterio: ¿Qué tan consistentemente supera su cuota?
- 145%+ = 10 puntos
- 130-144% = 8 puntos
- 120-129% = 6 puntos
- 110-119% = 4 puntos
- <110% = 2 puntos
Justificación: Importante para el objetivo de aumentar ventas 30%

**PASO 2: Evaluar Capacidad de Liderazgo (0-10 puntos)**
Criterio: ¿Tiene experiencia desarrollando vendedores?
- 3+ años liderando equipos = 10 puntos
- 2 años liderando equipos = 8 puntos
- 1 año liderando equipos = 6 puntos
- Mentor/co-líder sin reporte directo = 5 puntos
- Sin experiencia = 2 puntos
Justificación: Crítico para capacitar a los 3 vendedores nuevos

**PASO 3: Evaluar Fit Cultural (0-10 puntos)**
Criterio: ¿Qué tan bien colabora con el equipo?
- "Excelente" + comportamiento proactivo = 10 puntos
- "Excelente" sin detalles adicionales = 8 puntos
- "Muy respetado" = 8 puntos
- "Buena" = 6 puntos
- "A veces competitiva/conflictiva" = -2 puntos del puntaje base

**PASO 4: Evaluar Eficiencia Operativa (0-10 puntos)**
Criterio: Velocidad de cierre promedio
- 30 días = 10 puntos
- 35 días = 9 puntos
- 40 días = 8 puntos
- 45 días = 7 puntos
- 50 días = 6 puntos
- 60+ días = 4 puntos
Justificación: Procesos eficientes = más deals = más ventas

**PASO 5: Evaluar Antigüedad (0-10 puntos)**
Criterio: Años en la empresa
- 8+ años = 10 puntos
- 6-7 años = 9 puntos
- 5 años = 8 puntos
- 4 años = 7 puntos
- 3 años = 6 puntos
- 2 años = 5 puntos

---

**Formato de respuesta requerido:**

1. **Tabla Comparativa:**
   Crear una tabla con:
   - Columnas: Candidato | P1 | P2 | P3 | P4 | P5 | TOTAL
   - Filas: Un candidato por fila con sus puntajes

2. **Análisis individual (máximo 2 oraciones por candidato):**
   Para cada candidato, explicar brevemente sus fortalezas y debilidades principales

3. **Recomendación final (máximo 4 oraciones):**
   - ¿A quién recomendás?
   - ¿Por qué esta persona vs los demás?
   - ¿Cuál es su mayor fortaleza para los objetivos?

4. **Plan de desarrollo (máximo 3 oraciones):**
   - ¿Qué área debe mejorar el candidato recomendado?
   - ¿Cómo puede fortalecerse en esa área?"""

print("=== VERSIÓN CoT + RÚBRICA ===")
print(llamar_llm(mi_prompt_cot, max_tokens=800))

=== VERSIÓN CoT + RÚBRICA ===
##  Evaluación candidatos Jefe de Equipo de Ventas

**1. Tabla Comparativa:**

| Candidato | P1 | P2 | P3 | P4 | P5 | TOTAL |
|---|---|---|---|---|---|---|
| Martín | 10 | 2 | 8 | 7 | 9 | 36 |
| Carolina | 8 | 8 | 6 | 9 | 7 | 38 |
| Roberto | 8 | 5 | 8 | 4 | 10 | 35 |
| Lucía | 9 | 5 | 10 | 8 | 7 | 39 |
| Diego | 4 | 8 | 6 | 6 | 5 | 29 |
| Sofía | 8 | 6 | 10 | 8 | 8 | 38 |

**2. Análisis Individual:**

* **Martín:** Excelentes ventas, comparte conocimiento con el equipo, pero carece de experiencia formal en liderazgo.
* **Carolina:** Buena historial de ventas y experiencia previa como líder, pero puede ser competitiva.
* **Roberto:** Respetado por sus colegas, mentoría efectiva, pero su velocidad de cierre es lenta.
* **Lucía:** Excelente desempeño, iniciativa en capacitación y proactividad, pero necesita fortalecer su experiencia formal en liderazgo. 
* **Diego:** Buena capacidad de liderazgo en otro sector, pero su antigüedad en la empresa es baja.
* **S